## Model selection

Three regression models were compared using 3-fold cross-validation on the training dataset.

| Model | CV MAE | Validation mean position error |

| Ridge Regression | 16.42 | 49.80 |
| Tuned K-Nearest Neighbors | 2.26 | 14.37 |
| Tuned Random Forest | 2.88 | 12.45 |

Although KNN achieved the lowest random cross-validation MAE, the tuned Random Forest achieved the lowest error on the separate validation dataset.

Therefore, the tuned Random Forest regressor was selected as the final model for deployment.

In [8]:
def mean_position_error(y_true, predictions):
    errors = np.sqrt(
        ((y_true.to_numpy() - predictions) ** 2).sum(axis=1)
    )
    return errors.mean()

# Train and evaluate the Ridge baseline on validation data
ridge_final_model = models["Ridge Regression"]
ridge_final_model.fit(X_train, y_train)

ridge_predictions = ridge_final_model.predict(X_validation)

model_summary = pd.DataFrame({
    "Model": [
        "Ridge Regression",
        "Tuned K-Nearest Neighbors",
        "Tuned Random Forest"
    ],
    "Best CV MAE": [
        16.4206,
        2.2594,
        2.8843
    ],
    "Validation average coordinate MAE": [
        mean_absolute_error(y_validation, ridge_predictions),
        mean_absolute_error(y_validation, coordinate_predictions),
        mean_absolute_error(y_validation, rf_predictions)
    ],
    "Validation mean position error": [
        mean_position_error(y_validation, ridge_predictions),
        mean_position_error(y_validation, coordinate_predictions),
        mean_position_error(y_validation, rf_predictions)
    ]
})

model_summary.sort_values("Validation mean position error")

,Model,Best CV MAE,Validation average coordinate MAE,Validation mean position error
2,Tuned Random Forest,2.8843,8.060986,12.447902
1,Tuned K-Nearest Neighbors,2.2594,9.323327,14.368859
0,Ridge Regression,16.4206,32.695674,49.797803


In [7]:
rf_final_model = rf_search.best_estimator_

rf_predictions = rf_final_model.predict(X_validation)

rf_mae_per_coordinate = mean_absolute_error(
    y_validation,
    rf_predictions,
    multioutput="raw_values"
)

rf_position_errors = np.sqrt(
    ((y_validation.to_numpy() - rf_predictions) ** 2).sum(axis=1)
)

rf_r2_values = r2_score(
    y_validation,
    rf_predictions,
    multioutput="raw_values"
)

print(f"Longitude MAE: {rf_mae_per_coordinate[0]:.2f}")
print(f"Latitude MAE: {rf_mae_per_coordinate[1]:.2f}")
print(f"Mean position error: {rf_position_errors.mean():.2f}")
print(f"Median position error: {np.median(rf_position_errors):.2f}")
print(f"Longitude R²: {rf_r2_values[0]:.4f}")
print(f"Latitude R²: {rf_r2_values[1]:.4f}")

Longitude MAE: 8.65
Latitude MAE: 7.48
Mean position error: 12.45
Median position error: 7.97
Longitude R²: 0.9865
Latitude R²: 0.9703


In [6]:
from sklearn.model_selection import GridSearchCV

rf_parameters = {
    "max_depth": [None, 30],
    "min_samples_leaf": [1, 2]
}

rf_search = GridSearchCV(
    estimator=RandomForestRegressor(
        n_estimators=150,
        random_state=42,
        n_jobs=1
    ),
    param_grid=rf_parameters,
    scoring="neg_mean_absolute_error",
    cv=cv,
    n_jobs=-1
)

rf_search.fit(X_train, y_train)

print("Best settings:", rf_search.best_params_)
print(f"Best CV MAE: {-rf_search.best_score_:.4f}")

Best settings: {'max_depth': None, 'min_samples_leaf': 1}
Best CV MAE: 2.8843


In [5]:
import numpy as np
from sklearn.metrics import mean_absolute_error, r2_score

validation_df = pd.read_csv(data_dir / "validationData.csv")

X_validation = prepare_wifi_features(validation_df)
y_validation = validation_df[["LONGITUDE", "LATITUDE"]]

final_model = knn_search.best_estimator_
coordinate_predictions = final_model.predict(X_validation)

mae_per_coordinate = mean_absolute_error(
    y_validation,
    coordinate_predictions,
    multioutput="raw_values"
)

position_errors = np.sqrt(
    ((y_validation.to_numpy() - coordinate_predictions) ** 2).sum(axis=1)
)

r2_values = r2_score(
    y_validation,
    coordinate_predictions,
    multioutput="raw_values"
)

print(f"Longitude MAE: {mae_per_coordinate[0]:.2f}")
print(f"Latitude MAE: {mae_per_coordinate[1]:.2f}")
print(f"Mean position error: {position_errors.mean():.2f}")
print(f"Median position error: {np.median(position_errors):.2f}")
print(f"Longitude R²: {r2_values[0]:.4f}")
print(f"Latitude R²: {r2_values[1]:.4f}")


Longitude MAE: 9.63
Latitude MAE: 9.02
Mean position error: 14.37
Median position error: 7.90
Longitude R²: 0.9621
Latitude R²: 0.9295


In [4]:
from sklearn.model_selection import GridSearchCV

knn_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsRegressor(n_jobs=1))
])

knn_parameters = {
    "knn__n_neighbors": [1, 3, 5, 7, 9],
    "knn__weights": ["uniform", "distance"]
}

knn_search = GridSearchCV(
    estimator=knn_pipeline,
    param_grid=knn_parameters,
    scoring="neg_mean_absolute_error",
    cv=cv,
    n_jobs=-1
)

knn_search.fit(X_train, y_train)

print("Best settings:", knn_search.best_params_)
print(f"Best CV MAE: {-knn_search.best_score_:.4f}")

Best settings: {'knn__n_neighbors': 1, 'knn__weights': 'uniform'}
Best CV MAE: 2.2594


In [3]:
results = []

for model_name, model in models.items():
    scores = -cross_val_score(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring="neg_mean_absolute_error"
    )
    
    results.append({
        "Model": model_name,
        "CV MAE (mean)": scores.mean(),
        "CV MAE (std)": scores.std()
    })

results_df = pd.DataFrame(results).sort_values("CV MAE (mean)")
results_df

,Model,CV MAE (mean),CV MAE (std)
1,K-Nearest Neighbors,2.506486,0.050909
2,Random Forest,2.897121,0.012749
0,Ridge Regression,16.420612,0.070928


In [2]:
cv = KFold(n_splits=3, shuffle=True, random_state=42)

models = {
    "Ridge Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("ridge", Ridge(alpha=1.0))
    ]),
    
    "K-Nearest Neighbors": Pipeline([
        ("scaler", StandardScaler()),
        ("knn", KNeighborsRegressor(
            n_neighbors=5,
            weights="distance",
            n_jobs=-1
        ))
    ]),
    
    "Random Forest": RandomForestRegressor(
        n_estimators=100,
        random_state=42,
        n_jobs=-1
    )
}

In [1]:
import pandas as pd
from pathlib import Path

from sklearn.model_selection import KFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor

# Find the project root whether the notebook runs from the project root or notebooks/
project_root = Path.cwd()

if not (project_root / "data").exists():
    project_root = project_root.parent

data_dir = project_root / "data"

train_df = pd.read_csv(data_dir / "trainingData.csv")

def prepare_wifi_features(dataframe):
    """Select Wi-Fi columns and represent an undetected access point as -110."""
    wap_columns = [column for column in dataframe.columns if column.startswith("WAP")]
    return dataframe[wap_columns].replace(100, -110)

X_train = prepare_wifi_features(train_df)
y_train = train_df[["LONGITUDE", "LATITUDE"]]

print("Training inputs:", X_train.shape)
print("Coordinate targets:", y_train.shape)

Training inputs: (19937, 520)
Coordinate targets: (19937, 2)


# 06 — Coordinate Regression Model Comparison

This notebook compares Ridge Regression, K-Nearest Neighbors, and Random Forest Regression for predicting indoor longitude and latitude from Wi-Fi fingerprints.

Model selection uses 3-fold cross-validation on the training dataset only.
The separate validation dataset will be used once at the end for final evaluation.